# Bounding Volume Hierarchy (BVH) Example

This notebook demonstrates how to construct and query a Bounding Volume Hierarchy (BVH) on the GPU using `geocutool`. 
A BVH accelerates spatial queries like collision detection and ray tracing by organizing geometric primitives into a tree structure.

### Steps in this example:
1. **Mesh Generation**: We create a sphere mesh using Open3D and convert it to PyTorch tensors.
2. **AABB Computation**: We calculate the Axis-Aligned Bounding Box (AABB) for each triangle in the mesh.
3. **BVH Construction**: We build the BVH over these AABBs using `geocutool.data_structure.BVH`.
4. **Intersection Query**: We define a single query triangle, compute its AABB, and perform a broad-phase collision query against the BVH to find all overlapping bounding boxes.
5. **Visualization**: We render the base mesh, the query triangle, the intersected triangles, and their respective AABB wireframes using Plotly.

In [1]:
import torch
import numpy as np
import plotly.graph_objects as go
import open3d as o3d
import time

from geocutool.data_structure import BVH

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

sphere_mesh = o3d.geometry.TriangleMesh.create_sphere(radius=1.0, resolution=20)

vertices = torch.tensor(np.asarray(sphere_mesh.vertices), dtype=torch.float32, device=device)
triangles = torch.tensor(np.asarray(sphere_mesh.triangles), dtype=torch.long, device=device)

print(f"Mesh contains {triangles.shape[0]} triangles.")

tri_verts = vertices[triangles]

aabb_mins, _ = torch.min(tri_verts, dim=1)
aabb_maxs, _ = torch.max(tri_verts, dim=1)

print("Building GPU BVH...")
start_time = time.perf_counter()
bvh = BVH(aabb_mins, aabb_maxs)
torch.cuda.synchronize()
print(f"BVH Built in {(time.perf_counter() - start_time) * 1000:.2f} ms")

query_verts = torch.tensor([
    [-1.0, 0.5, -0.3],
    [ 1.0, -0.5, -0.3],
    [ 0.0, 0.0,  2.0]
], dtype=torch.float32, device=device)

query_min, _ = torch.min(query_verts, dim=0, keepdim=True)
query_max, _ = torch.max(query_verts, dim=0, keepdim=True)

print("Querying BVH...")
start_time = time.perf_counter()

q_ids, intersected_tri_ids = bvh.query(query_min, query_max)
torch.cuda.synchronize()

num_hits = intersected_tri_ids.size(0)
print(f"Query Completed in {(time.perf_counter() - start_time) * 1000:.2f} ms")
print(f"Found {num_hits} broad-phase triangle intersections!")

v_np = vertices.cpu().numpy()
t_np = triangles.cpu().numpy()
intersected_idx = intersected_tri_ids.cpu().numpy()
q_v_np = query_verts.cpu().numpy()

fig = go.Figure()

fig.add_trace(go.Mesh3d(
    x=v_np[:, 0], y=v_np[:, 1], z=v_np[:, 2],
    i=t_np[:, 0], j=t_np[:, 1], k=t_np[:, 2],
    color='lightgray',
    opacity=0.5,
    name='Base Mesh',
    hoverinfo='skip'
))

if num_hits > 0:
    hit_triangles = t_np[intersected_idx]
    fig.add_trace(go.Mesh3d(
        x=v_np[:, 0], y=v_np[:, 1], z=v_np[:, 2],
        i=hit_triangles[:, 0], j=hit_triangles[:, 1], k=hit_triangles[:, 2],
        color='red',
        opacity=1.0,
        name='Intersected AABBs',
        flatshading=True
    ))
    
    hit_mins = aabb_mins.cpu().numpy()[intersected_idx]
    hit_maxs = aabb_maxs.cpu().numpy()[intersected_idx]

    box_x, box_y, box_z = [], [], []
    
    # Build line segments for the boxes. We insert `None` to break the continuous 
    # Plotly line between different edges/boxes so they don't connect to each other.
    for min_pt, max_pt in zip(hit_mins, hit_maxs):
        x0, y0, z0 = min_pt
        x1, y1, z1 = max_pt

        # Bottom face ring
        box_x.extend([x0, x1, x1, x0, x0, None])
        box_y.extend([y0, y0, y1, y1, y0, None])
        box_z.extend([z0, z0, z0, z0, z0, None])

        # Top face ring
        box_x.extend([x0, x1, x1, x0, x0, None])
        box_y.extend([y0, y0, y1, y1, y0, None])
        box_z.extend([z1, z1, z1, z1, z1, None])

        # 4 Vertical Pillars connecting top and bottom
        box_x.extend([x0, x0, None, x1, x1, None, x1, x1, None, x0, x0, None])
        box_y.extend([y0, y0, None, y0, y0, None, y1, y1, None, y1, y1, None])
        box_z.extend([z0, z1, None, z0, z1, None, z0, z1, None, z0, z1, None])

    fig.add_trace(go.Scatter3d(
        x=box_x, y=box_y, z=box_z,
        mode='lines',
        line=dict(color='orange', width=2),
        name='AABB Wireframes',
        hoverinfo='skip'
    ))

fig.add_trace(go.Mesh3d(
    x=q_v_np[:, 0], y=q_v_np[:, 1], z=q_v_np[:, 2],
    i=[0], j=[1], k=[2],
    color='blue',
    opacity=0.6,
    name='Query Triangle'
))

q_min_np = query_min.cpu().numpy()[0]
q_max_np = query_max.cpu().numpy()[0]

qbox_x, qbox_y, qbox_z = [], [], []
qx0, qy0, qz0 = q_min_np
qx1, qy1, qz1 = q_max_np

# Bottom
qbox_x.extend([qx0, qx1, qx1, qx0, qx0, None])
qbox_y.extend([qy0, qy0, qy1, qy1, qy0, None])
qbox_z.extend([qz0, qz0, qz0, qz0, qz0, None])
# Top
qbox_x.extend([qx0, qx1, qx1, qx0, qx0, None])
qbox_y.extend([qy0, qy0, qy1, qy1, qy0, None])
qbox_z.extend([qz1, qz1, qz1, qz1, qz1, None])
# Pillars
qbox_x.extend([qx0, qx0, None, qx1, qx1, None, qx1, qx1, None, qx0, qx0, None])
qbox_y.extend([qy0, qy0, None, qy0, qy0, None, qy1, qy1, None, qy1, qy1, None])
qbox_z.extend([qz0, qz1, None, qz0, qz1, None, qz0, qz1, None, qz0, qz1, None])

fig.add_trace(go.Scatter3d(
    x=qbox_x, y=qbox_y, z=qbox_z,
    mode='lines',
    line=dict(color='cyan', width=4, dash='dash'),
    name='Query AABB',
    hoverinfo='skip'
))

fig.update_layout(
    title="BVH Broad-Phase AABB Intersection",
    scene=dict(
        xaxis=dict(showbackground=False, visible=False),
        yaxis=dict(showbackground=False, visible=False),
        zaxis=dict(showbackground=False, visible=False),
        aspectmode='data'
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Running on: cuda
Mesh contains 1520 triangles.
Building GPU BVH...
BVH Built in 1.05 ms
Querying BVH...
Query Completed in 2.54 ms
Found 578 broad-phase triangle intersections!
